# KrishiMitra-RS — end-to-end walkthrough

Crop-type mapping, phenology-aware moisture-stress detection and FAO-56 irrigation advisory
from optical + SAR satellite data. This notebook runs the whole pipeline stage by stage and
shows the intermediate products.

> Runs on the built-in **simulated** pilot — no downloads, no GPU. Set `data.source: gee` in
> `config/pilot_area.yaml` to run the same code on real Sentinel-1/2.


In [ ]:
import sys; sys.path.insert(0, 'src')
import numpy as np, matplotlib.pyplot as plt
from krishimitra_rs.config import load_config
cfg = load_config()
print(cfg.pilot['name'])
print('season:', cfg.start_date, '->', cfg.end_date, '|', cfg.n_timesteps, '8-day composites')
print('crops:', cfg.crop_names)


## 1 · Analysis-ready data cube (simulate or ingest)
Optical (red/nir/green/swir1), SAR (VV/VH), a cloud mask, and daily meteorology.


In [ ]:
from krishimitra_rs.data.simulate import simulate_cube
cube = simulate_cube(cfg)
print('optical NDVI-band stack:', cube.optical['nir'].shape, '| cloud frac:', round(float(np.isnan(cube.optical['red']).mean()),3))
print('SAR VV stack:', cube.sar['vv'].shape, '(no NaNs — all-weather)')


### NDVI phenology by crop
The multi-temporal spectral signature — each crop has a distinct green-up / peak / senescence.


In [ ]:
nir, red = cube.optical['nir'], cube.optical['red']
ndvi = (nir-red)/(nir+red+1e-6)
plt.figure(figsize=(10,4))
for c in cfg.crops:
    m = cube.labels==c['code']
    if m.any(): plt.plot([d.isoformat()[5:] for d in cube.dates], np.nanmean(ndvi[:,m],1), marker='o', ms=3, label=c['name'], color=c['color'])
plt.xticks(rotation=45); plt.ylabel('NDVI'); plt.legend(ncol=3); plt.title('NDVI phenology'); plt.tight_layout()


## 2 · Feature extraction
Indices (NDVI/EVI/NDWI/RVI/CR), GLCM texture, and phenology metrics → 227 features/pixel.


In [ ]:
from krishimitra_rs.features.build import build_feature_stack
fs = build_feature_stack(cube)
print('feature matrix:', fs.X.shape, '| e.g.', fs.names[:4], '...', fs.names[-3:])


## 3 · Crop-type classification (RF + XGBoost, field-disjoint validation)


In [ ]:
from krishimitra_rs.models.crop_classifier import classify_crops
crop = classify_crops(cube, fs, cfg)
m = crop.metrics[crop.best_model]
print(f'best={crop.best_model}  OA={m["overall_accuracy"]:.1%}  kappa={m["kappa"]:.2f}')
for k,v in m['per_class'].items(): print(f'  {k:10} F1={v["f1"]:.2f}')


In [ ]:
plt.imshow(crop.crop_map, cmap='tab10'); plt.title('Crop-type map'); plt.axis('off')


## 4 · Growth stage + phenology-aware moisture stress
VCI (optical) + SMI (SAR), fused and weighted by growth stage (flowering most sensitive).


In [ ]:
from krishimitra_rs.advisory.phenology_stage import detect_growth_stage
from krishimitra_rs.models.stress import detect_moisture_stress
stage = detect_growth_stage(fs.indices['ndvi'], fs.pheno)
stress = detect_moisture_stress(cube, fs, cfg, stage, crop.crop_map)
print('stress vs latent truth:', stress.validation)
plt.imshow(stress.season_peak_class, cmap='RdYlGn_r'); plt.title('Season peak stress'); plt.axis('off')


## 5 · FAO-56 water balance → irrigation advisory


In [ ]:
from krishimitra_rs.advisory.phenology_stage import kc_from_detected_stage
from krishimitra_rs.advisory.water_balance import water_balance
from krishimitra_rs.advisory.irrigation import generate_advisory
kc = kc_from_detected_stage(stage, fs.pheno, crop.crop_map, cfg)
wb = water_balance(cube, kc, fs.indices, crop.crop_map, cfg)
adv = generate_advisory(cube, wb, stage, crop.crop_map, cfg)
print(adv.command_area_summary)
plt.imshow(adv.latest_class, cmap='RdYlGn_r'); plt.title('Irrigation advisory (peak demand)'); plt.axis('off')


## 6 · Validation report


In [ ]:
from krishimitra_rs.validation.metrics import build_validation_report
import json
rep = build_validation_report(crop, stress, wb, adv, cube, cfg)
print(json.dumps(rep['classification'], indent=2)[:600])
print('advisory credible:', rep['advisory']['consistent'], '| corr vs truth:', rep['advisory'].get('advisory_vs_trueKs_corr'))


---
That's the full chain. To run everything and write all figures/tables at once:
```python
from krishimitra_rs.pipeline import run
res = run()   # -> outputs/
```
